In [1]:
# %% [markdown]
# # 玩具车拆装场景事件分割Pipeline
# 
# ## 🎯 任务目标
# - **原子事件**: 拆/装单个螺丝、取下/安装单个部件 (3-8秒)
# - **中层事件**: 拆/装左轮、右轮、车身等 (15-30秒)
# - **顶层事件**: 拆解玩具车 vs 组装玩具车 (1-2.5分钟)

# %% [markdown]
# ## 🔧 环境准备

# %%
import os, subprocess, shutil
from pathlib import Path
import numpy as np
import torch
import clip
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import json

print(f"✅ PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# %% [markdown]
# ## 📹 配置参数

# %%
# ========== 基础配置 ==========
VIDEO_PATH = "/media/wang/New Volume1/josie/gebd/C10390_rgb.mp4"
OUTPUT_DIR = "pipeline_output"
FPS = 10
BATCH_SIZE = 64
CLEAN_FRAMES = False
REUSE_EXISTING_FRAMES = True

# ========== 分割策略 ==========
# 完整增强版分割流程：
# 1. 不对称滑动窗口（自适应参数搜索）
#    - 窗口大小: 10/12/15 帧
#    - 阈值: 0.72/0.74/0.76/0.80
#    - 相似度方法: cosine
# 2. 传统差分峰值检测（85分位，Savitzky-Golay滤波）
# 3. 融合两路候选（0.3s最小间隔去重）
# 4. 边界精修（±0.3s范围内选最大梯度点）
# 5. 质量兜底（score < 0.5时选择最优单一方法）

# ========== 层次结构配置 ==========
HIERARCHY_CONFIG = {
    'sigma': 30.0,  # 时间核带宽（秒）- 减小以增强局部性
    'cut_levels': {
        'atomic': None,           # 原子级（不切割）
        'component': 70,          # 部件级（如单个轮子）
        'side': 85,               # 侧面级（如左侧/右侧）
        'activity': 95            # 活动级（拆解/组装）
    }
}

# ========== 路径设置 ==========
video_path = Path(VIDEO_PATH)
video_name = video_path.stem
output_dir = Path(OUTPUT_DIR) / video_name
frame_dir = output_dir / "frames"
frame_dir.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n🎬 视频: {video_name}")
print(f"📁 输出: {output_dir}")
print(f"🔧 设备: {device}, FPS: {FPS}")
print(f"🎯 策略: 完整增强版（不对称窗口 + 差分峰值 + 融合 + 精修）")

# %% [markdown]
# ## Step 1: 帧提取

# %%
print("\n" + "="*70)
print("[Step 1/3] 帧提取")
print("="*70)

if CLEAN_FRAMES:
    print("清理旧帧...")
    for p in frame_dir.glob('*.jpg'):
        p.unlink()

frame_files = sorted(list(frame_dir.glob('*.jpg')))
if frame_files and REUSE_EXISTING_FRAMES:
    print(f"✅ 复用已有 {len(frame_files)} 帧")
else:
    print("使用FFmpeg提取帧...")
    cmd = [
        'ffmpeg', '-y', '-v', 'error',
        '-i', str(video_path),
        '-vf', f'fps={FPS}',
        '-q:v', '2',
        str(frame_dir / '%06d.jpg')
    ]
    subprocess.run(cmd, check=True)
    frame_files = sorted(list(frame_dir.glob('*.jpg')))

num_frames = len(frame_files)
print(f"✅ 共 {num_frames} 帧 ({num_frames/FPS:.2f}秒)")

# %% [markdown]
# ## Step 2: CLIP编码

# %%
print("\n" + "="*70)
print("[Step 2/3] CLIP编码")
print("="*70)

print("加载CLIP模型...")
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

feats_list = []
with torch.no_grad():
    for i in tqdm(range(0, num_frames, BATCH_SIZE), desc="编码帧"):
        images = []
        for f in frame_files[i:i+BATCH_SIZE]:
            try:
                with Image.open(f) as img:
                    images.append(preprocess(img.convert("RGB")))
            except Exception as e:
                print(f"⚠️  跳过损坏帧: {f}")
                continue
        
        if images:
            image_input = torch.stack(images).to(device)
            features = model.encode_image(image_input)
            features = features / features.norm(dim=-1, keepdim=True)
            feats_list.append(features.cpu().numpy())

features = np.concatenate(feats_list, axis=0)
print(f"✅ 特征矩阵: {features.shape}")

# %% [markdown]
# ## Step 3: 定义优化的分割算法

# %%
def compute_window_similarity(window_a, window_b, method='cosine'):
    """计算两个窗口的相似度"""
    if method == 'cosine':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        na, nb = np.linalg.norm(mean_a), np.linalg.norm(mean_b)
        if na == 0 or nb == 0:
            return 0.0
        return float(np.dot(mean_a, mean_b) / (na * nb))
    elif method == 'l2':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        return float(np.exp(-np.linalg.norm(mean_a - mean_b)))
    elif method == 'correlation':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        if np.std(mean_a) == 0 or np.std(mean_b) == 0:
            return 1.0 if np.allclose(mean_a, mean_b) else 0.0
        r = np.corrcoef(mean_a, mean_b)[0, 1]
        return float(r) if not np.isnan(r) else 0.0
    return 0.0


def sw_asymmetric(feats, window_size=10, step_reset=1, threshold=0.8,
                  similarity_method='cosine', min_segment_frames=10):
    """不对称滑动窗口分割
    
    右窗固定=window_size；左窗从window_size起，每次+1，
    达上限(3*window)或触发切分则重置；reset步长=1。
    """
    T = feats.shape[0]
    right_size = int(window_size)
    left_init = int(window_size)
    grow = 1
    max_left = right_size * 3
    
    boundaries, similarities, positions = [], [], []
    pos, left_len = 0, left_init
    
    while pos + left_len + right_size <= T:
        left_start = pos
        left_end = pos + left_len
        right_end = left_end + right_size
        
        a = feats[left_start:left_end]
        b = feats[left_end:right_end]
        sim = compute_window_similarity(a, b, similarity_method)
        
        similarities.append(float(sim))
        positions.append(left_end)
        
        if sim < threshold:
            mid = left_end
            if (not boundaries) or (mid - boundaries[-1] >= min_segment_frames):
                boundaries.append(int(mid))
                pos = mid
                left_len = left_init
            else:
                left_len = min(left_len + grow, max_left)
                if (pos + left_len + right_size > T) or (left_len >= max_left):
                    pos += step_reset
                    left_len = left_init
        else:
            left_len = min(left_len + grow, max_left)
            if (pos + left_len + right_size > T) or (left_len >= max_left):
                pos += step_reset
                left_len = left_init
    
    return boundaries, similarities, positions


def traditional_diff_peaks(feats_scaled, fps):
    """传统差分峰值检测（85分位），预去重间隔 0.3s"""
    div_boundaries = []
    try:
        from scipy.signal import savgol_filter
        diffs = feats_scaled[1:] - feats_scaled[:-1]
        l2 = np.linalg.norm(diffs, axis=1)
        if len(l2) >= 5:
            win = min(7, len(l2) // 4)
            win += 1 if win % 2 == 0 else 0
            if win >= 3:
                l2_s = savgol_filter(l2, win, 1, mode='mirror')
                thr = np.percentile(l2_s, 85)
                cand = np.where(l2_s > thr)[0]
                gap = max(1, int(0.3 * fps))  # 0.3s
                for c in cand:
                    if not div_boundaries or c - div_boundaries[-1] >= gap:
                        div_boundaries.append(int(c))
    except Exception:
        pass
    return div_boundaries


def refine_boundary(b, feats, fps, radius=None):
    """精修边界：±0.3s 内选帧间差分 L2 平滑曲线的局部最大作为边界"""
    if radius is None:
        radius = max(2, int(0.3 * fps))  # 0.3s
    L = feats.shape[0]
    start = max(2, b - radius)
    end = min(L - 2, b + radius)
    diffs = feats[1:] - feats[:-1]
    l2 = np.linalg.norm(diffs, axis=1)
    l2_smooth = np.convolve(l2, np.ones(3)/3, mode='same') if len(l2) >= 3 else l2
    idx = np.argmax(l2_smooth[start:end+1])
    return int(start + idx)


def post_fuse_and_refine(boundaries_a, boundaries_b, feats_scaled, fps):
    """融合两路候选，采用 0.3s 的最小间隔去重，再做精修（同样 0.3s 的最小间隔）"""
    MIN_DIST = max(1, int(0.3 * fps))  # 0.3s
    merged = sorted(set(boundaries_a + boundaries_b))
    final = []
    for b in merged:
        if not final or b - final[-1] >= MIN_DIST:
            final.append(int(b))
    # 精修
    refined = []
    for b in final:
        rb = refine_boundary(b, feats_scaled, fps)
        if not refined or rb - refined[-1] >= MIN_DIST:
            refined.append(rb)
    return refined


def evaluate_segmentation_quality(boundaries, total_frames, fps=10):
    """评估分割质量"""
    if len(boundaries) == 0:
        return 0.7 if total_frames >= fps * 5 else 0.5
    
    seg_lens, prev = [], 0
    for b in boundaries + [total_frames]:
        seg_lens.append(b - prev)
        prev = b
    
    durs = [l / fps for l in seg_lens]
    mn, av, n, sd = min(durs), np.mean(durs), len(durs), np.std(durs)
    
    score = 1.0
    if mn < 1.0:
        score *= 0.2
    elif mn < 1.5:
        score *= 0.7
    
    if n > 6:
        score *= 0.3
    elif n > 4:
        score *= 0.8
    
    if av < 1.5:
        score *= 0.4
    elif av > 6.0:
        score *= 0.7
    
    if sd > av * 0.8:
        score *= 0.8
    
    return score


def adaptive_asymmetric_search(feats_scaled, fps):
    """自适应不对称滑动窗口参数搜索"""
    # 更密网格：窗口 10/12/15；阈值 0.72/0.74/0.76/0.80
    window_sizes = [10, 12, 15]
    thresholds = [0.72, 0.74, 0.76, 0.80]
    methods = ['cosine']
    
    best_boundaries, best_score, best_params = [], 0.0, {}
    
    print(f"\n🔍 自适应搜索空间:")
    print(f"   窗口大小: {window_sizes} 帧 = {[w/fps for w in window_sizes]} 秒")
    print(f"   相似度阈值: {thresholds}")
    print(f"   相似度方法: {methods}")
    
    for ws in window_sizes:
        for thr in thresholds:
            for m in methods:
                try:
                    min_seg = max(fps, ws)
                    boundaries, sims, pos = sw_asymmetric(
                        feats_scaled, window_size=ws, step_reset=1,
                        threshold=thr, similarity_method=m, min_segment_frames=min_seg
                    )
                    score = evaluate_segmentation_quality(boundaries, feats_scaled.shape[0], fps)
                    
                    if score > best_score:
                        best_boundaries, best_score = boundaries, score
                        best_params = {
                            'window_size': ws,
                            'threshold': thr,
                            'method': m,
                            'num_boundaries': len(boundaries),
                            'score': score
                        }
                except Exception:
                    continue
    
    return best_boundaries, best_params


def enhanced_runner(feats, fps):
    """完整的增强分割执行器（不对称窗口 + 差分峰值 + 融合 + 精修）"""
    print("\n" + "-"*70)
    
    # 标准化特征
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(feats)
    
    # 1) 不对称滑动窗口
    print("🔹 步骤1: 不对称滑动窗口分割...")
    sw_boundaries, sw_params = adaptive_asymmetric_search(feats_scaled, fps)
    print(f"   ✓ 找到 {len(sw_boundaries)} 个边界")
    print(f"   ✓ 最佳参数: ws={sw_params['window_size']}, thr={sw_params['threshold']:.2f}")
    print(f"   ✓ 质量分数: {sw_params['score']:.3f}")
    
    # 2) 传统差分峰值检测（85分位），预去重间隔 0.3s
    print("\n🔹 步骤2: 差分峰值检测...")
    div_boundaries = traditional_diff_peaks(feats_scaled, fps)
    print(f"   ✓ 找到 {len(div_boundaries)} 个边界")
    
    # 3) 融合 + 精修
    print("\n🔹 步骤3: 融合与精修...")
    final = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled, fps)
    print(f"   ✓ 融合后: {len(final)} 个边界")
    
    # 4) 质量兜底
    score = evaluate_segmentation_quality(final, feats.shape[0], fps)
    if score < 0.5:
        if len(sw_boundaries) <= len(div_boundaries):
            final = sw_boundaries
            method_used = 'asymmetric_only'
        else:
            final = div_boundaries
            method_used = 'divergence_only'
    else:
        method_used = 'asymmetric_hybrid_div'
    
    # 统计信息
    num_segments = len(final) + 1
    avg_duration = feats.shape[0] / num_segments / fps
    
    print(f"\n📊 最终统计:")
    print(f"   边界数: {len(final)}")
    print(f"   片段数: {num_segments}")
    print(f"   平均片段长度: {avg_duration:.1f}秒")
    print(f"   最终质量分数: {score:.3f}")
    print(f"   使用方法: {method_used}")
    print("-"*70)
    
    # 构建参数字典
    params = {
        **sw_params,
        'method_used': method_used,
        'sw_boundaries': len(sw_boundaries),
        'div_boundaries': len(div_boundaries),
        'final_boundaries': len(final),
        'final_score': score,
        'num_segments': num_segments,
        'avg_segment_sec': avg_duration
    }
    
    return final, params


print("✅ 分割算法已加载（完整增强版：不对称窗口 + 差分峰值 + 融合 + 精修）")

# %% [markdown]
# ## Step 3: 执行原子分割

# %%
print("\n" + "="*70)
print("[Step 3/3] 原子级事件分割")
print("="*70)

# 执行完整的增强分割（不对称窗口 + 差分峰值 + 融合 + 精修）
boundaries, params = enhanced_runner(features, FPS)

results = {'asymmetric': {'boundaries': boundaries, 'params': params}}

print("\n✅ 原子分割完成")

# %% [markdown]
# ## 保存原子分割结果

# %%
def create_segments(boundaries, T):
    """从边界列表创建片段列表"""
    segs = []
    prev = 0
    for b in sorted(boundaries):
        b = int(b)
        if b > prev:
            segs.append([int(prev), int(b)])
            prev = b
    if prev < T:
        segs.append([int(prev), int(T)])
    return segs

T = features.shape[0]
segs = create_segments(boundaries, T)

json_data = {
    'video_name': video_name,
    'num_frames': int(T),
    'fps': float(FPS),
    'video_duration': round(T / FPS, 3),
    'num_boundaries': len(boundaries),
    'num_segments': len(segs),
    'boundaries_frame_indices': [int(b) for b in boundaries],
    'boundaries_timestamps': [round(b / FPS, 3) for b in boundaries],
    'segments_timestamps': [[round(s[0]/FPS, 3), round(s[1]/FPS, 3)] for s in segs],
    'method': 'enhanced_asymmetric_hybrid',
    'parameters': params
}

out_path = output_dir / f"{video_name}_atomic_segmentation.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

print(f"✅ 原子分割JSON: {out_path}")

# 可视化原子分割
fig, ax = plt.subplots(figsize=(20, 3))
colors = plt.cm.Set3(np.linspace(0, 1, len(segs)))

for idx, (start, end) in enumerate(segs):
    ax.barh(0, end-start, left=start/FPS, height=0.8, 
            color=colors[idx % len(colors)], edgecolor='black', linewidth=0.5)
    # 添加时间标注 (mm:ss-mm:ss)
    seg_center = (start + end) / 2 / FPS
    start_time = start / FPS
    end_time = end / FPS
    
    start_min = int(start_time // 60)
    start_sec = int(start_time % 60)
    end_min = int(end_time // 60)
    end_sec = int(end_time % 60)
    
    time_label = f'{start_min:02d}:{start_sec:02d}-{end_min:02d}:{end_sec:02d}'
    
    if (end - start) / FPS > 2:  # 只在较长片段上显示时间
        ax.text(seg_center, 0, time_label, ha='center', va='center', 
                fontsize=6, color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))

ax.set_xlim(0, T/FPS)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_title(f'Atomic-level Segmentation - {len(segs)} Segments', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
atomic_viz_path = output_dir / f"{video_name}_atomic_segments.png"
plt.savefig(atomic_viz_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ 原子分割可视化: {atomic_viz_path}")

# %% [markdown]
# ## Phase I: 构建语义一致性图谱

# %%
print("\n" + "="*70)
print("Phase I: 构建语义一致性图谱 (SCM)")
print("="*70)

K = len(boundaries) + 1  # 原子单元数量
print(f"原子单元数量: {K}")


def extract_atomic_features(features, boundaries, fps):
    """从CLIP特征中提取每个原子单元的代表性特征"""
    T = features.shape[0]
    segments = []
    
    # 构建片段列表
    prev = 0
    for b in boundaries:
        if b > prev:
            segments.append((prev, b))
            prev = b
    if prev < T:
        segments.append((prev, T))
    
    atomic_features = []
    atomic_centers = []
    
    for start, end in segments:
        # 计算片段的平均特征
        segment_feats = features[start:end]
        mean_feat = np.mean(segment_feats, axis=0)
        
        # L2归一化
        mean_feat = mean_feat / (np.linalg.norm(mean_feat) + 1e-8)
        atomic_features.append(mean_feat)
        
        # 计算时间中心（秒）
        center_time = (start + end) / 2.0 / fps
        atomic_centers.append(center_time)
    
    return np.array(atomic_features), np.array(atomic_centers)


# 提取原子单元特征
atomic_features, atomic_centers = extract_atomic_features(features, boundaries, FPS)

print(f"原子特征矩阵: {atomic_features.shape}")
print(f"时间中心范围: {atomic_centers[0]:.2f}s ~ {atomic_centers[-1]:.2f}s")


def build_semantic_consistency_map(atomic_features, atomic_centers, sigma):
    """构建语义一致性图谱 (SCM)"""
    K = len(atomic_features)
    M = np.zeros((K, K))
    
    for i in range(K):
        for j in range(K):
            # 1. 语义核：余弦相似度
            K_sem = np.dot(atomic_features[i], atomic_features[j])
            
            # 2. 时间核：高斯RBF
            time_diff = atomic_centers[i] - atomic_centers[j]
            K_temp = np.exp(- (time_diff ** 2) / (2 * sigma ** 2))
            
            # 3. 混合核
            M[i, j] = K_sem * K_temp
    
    return M


# 构建SCM
sigma = HIERARCHY_CONFIG['sigma']
SCM = build_semantic_consistency_map(atomic_features, atomic_centers, sigma)

print(f"\n✅ SCM构建完成")
print(f"   矩阵大小: {SCM.shape}")
print(f"   时间核带宽 σ: {sigma}秒")
print(f"   亲和度范围: [{SCM.min():.4f}, {SCM.max():.4f}]")

# %%
# 可视化SCM
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(SCM, cmap='hot', aspect='auto', interpolation='nearest')
axes[0].set_title(
    f'Semantic Consistency Map\nσ={sigma}s, {K} Atomic Units',
    fontsize=12, fontweight='bold'
)
axes[0].set_xlabel('Atomic Unit Index', fontsize=10)
axes[0].set_ylabel('Atomic Unit Index', fontsize=10)
plt.colorbar(im1, ax=axes[0], label='Affinity Score')

band_width = min(15, K//2)
band_mask = np.abs(np.arange(K)[:, None] - np.arange(K)[None, :]) <= band_width
SCM_band = SCM.copy()
SCM_band[~band_mask] = 0

im2 = axes[1].imshow(SCM_band, cmap='hot', aspect='auto', interpolation='nearest')
axes[1].set_title(
    f'Band Diagonal Structure (±{band_width} Units)',
    fontsize=12, fontweight='bold'
)
axes[1].set_xlabel('Atomic Unit Index', fontsize=10)
axes[1].set_ylabel('Atomic Unit Index', fontsize=10)
plt.colorbar(im2, ax=axes[1], label='Affinity Score')

plt.tight_layout()
scm_path = output_dir / f"{video_name}_SCM.png"
plt.savefig(scm_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ SCM可视化: {scm_path}")

# %% [markdown]
# ## Phase II: 层次拓扑解析

# %%
print("\n" + "="*70)
print("Phase II: 层次拓扑解析 (Ward's HAC)")
print("="*70)


def similarity_to_distance(similarity_matrix):
    """将相似度矩阵转换为距离矩阵"""
    sim_clipped = np.clip(similarity_matrix, 0, 1)
    distance = np.sqrt(2 * (1 - sim_clipped))
    np.fill_diagonal(distance, 0)
    return distance


distance_matrix = similarity_to_distance(SCM)
condensed_distance = squareform(distance_matrix, checks=False)

# Ward's层次聚类
linkage_matrix = linkage(condensed_distance, method='ward')
merge_distances = linkage_matrix[:, 2]

print(f"✅ 聚类完成")
print(f"   合并距离范围: [{merge_distances.min():.4f}, {merge_distances.max():.4f}]")

# %%
# 可视化树状图
fig, ax = plt.subplots(figsize=(18, 8))
dendrogram_data = dendrogram(
    linkage_matrix,
    ax=ax,
    color_threshold=np.percentile(merge_distances, 85),
    above_threshold_color='gray',
    leaf_font_size=6
)
ax.set_title(
    f'Hierarchical Dendrogram - {video_name} ({K} Atomic Units)',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel('Atomic Unit Index', fontsize=12)
ax.set_ylabel('Ward Distance', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# 添加分割线
cut_levels = HIERARCHY_CONFIG['cut_levels']
for level_name, percentile in cut_levels.items():
    if percentile is not None:
        threshold = np.percentile(merge_distances, percentile)
        ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.text(K*0.98, threshold, f'{level_name} ({percentile}%)', 
                verticalalignment='bottom', fontsize=9, color='red')

plt.tight_layout()
dendrogram_path = output_dir / f"{video_name}_dendrogram.png"
plt.savefig(dendrogram_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ 树状图: {dendrogram_path}")

# %% [markdown]
# ## 多粒度分割

# %%
print("\n" + "="*70)
print("多粒度事件分割")
print("="*70)

hierarchical_segmentations = {}
cut_levels = HIERARCHY_CONFIG['cut_levels']

for level_name, percentile in cut_levels.items():
    if percentile is None:
        # 原子级不切割
        hierarchical_segmentations[level_name] = {
            'threshold': 0.0,
            'num_events': K,
            'labels': np.arange(1, K+1)
        }
        print(f"{level_name:12s}: {K:3d} 个原子单元")
    else:
        threshold = np.percentile(merge_distances, percentile)
        cluster_labels = fcluster(linkage_matrix, threshold, criterion='distance')
        num_clusters = len(np.unique(cluster_labels))
        
        hierarchical_segmentations[level_name] = {
            'threshold': threshold,
            'num_events': num_clusters,
            'labels': cluster_labels
        }
        
        print(f"{level_name:12s}: {num_clusters:3d} 个事件 "
              f"(平均 {K/num_clusters:.1f} 原子/事件, 阈值={threshold:.3f})")

# %%
# 可视化多粒度分割
fig, axes = plt.subplots(len(hierarchical_segmentations), 1, 
                         figsize=(20, 2*len(hierarchical_segmentations)))

level_names = ['atomic', 'component', 'side', 'activity']

for idx, (level_name, ax) in enumerate(zip(level_names, axes)):
    level_data = hierarchical_segmentations[level_name]
    labels = level_data['labels']
    num_events = level_data['num_events']
    colors = plt.cm.tab20(np.linspace(0, 1, num_events))
    
    # 绘制每个原子单元
    for i in range(K):
        cluster_id = labels[i] - 1
        color = colors[cluster_id % len(colors)]
        ax.barh(0, 1, left=i, height=0.8, color=color, edgecolor='white', linewidth=0.5)
    
    # 添加事件分割线和时间标注 (mm:ss-mm:ss)
    event_boundaries = []
    for i in range(1, K):
        if labels[i] != labels[i-1]:
            event_boundaries.append(i)
    
    # 在每个事件上标注起始和结束时间
    prev_boundary = 0
    for boundary in event_boundaries + [K]:
        event_center_idx = (prev_boundary + boundary) / 2
        
        # 计算事件的起始和结束时间（秒）
        start_idx = int(prev_boundary) if prev_boundary < len(atomic_centers) else len(atomic_centers) - 1
        end_idx = int(boundary - 1) if boundary - 1 < len(atomic_centers) else len(atomic_centers) - 1
        
        start_time = atomic_centers[start_idx]
        end_time = atomic_centers[end_idx]
        
        # 格式化为 mm:ss
        start_min = int(start_time // 60)
        start_sec = int(start_time % 60)
        end_min = int(end_time // 60)
        end_sec = int(end_time % 60)
        
        time_label = f'{start_min:02d}:{start_sec:02d}-{end_min:02d}:{end_sec:02d}'
        
        # 只在较大的事件上显示时间
        if (boundary - prev_boundary) > K * 0.03:  # 事件占比超过3%
            ax.text(event_center_idx, 0, time_label, ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))
        
        # 绘制分割线
        if boundary < K:
            ax.axvline(x=boundary, color='black', linestyle='-', linewidth=2, alpha=0.8)
        
        prev_boundary = boundary
    
    ax.set_xlim(0, K)
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.set_xlabel('Atomic Unit Index', fontsize=10)
    ax.set_title(
        f'{level_name.upper()} Level - {num_events} Events',
        fontsize=11, fontweight='bold'
    )
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
hierarchy_path = output_dir / f"{video_name}_hierarchy.png"
plt.savefig(hierarchy_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ 层次分割可视化: {hierarchy_path}")

# %% [markdown]
# ## 保存层次结构结果

# %%
# 保存层次结构为JSON
hierarchy_json = {
    'video_name': video_name,
    'num_atomic_units': int(K),
    'scm_sigma': float(sigma),
    'linkage_method': 'ward',
    'hierarchical_levels': {}
}

for level_name, level_data in hierarchical_segmentations.items():
    hierarchy_json['hierarchical_levels'][level_name] = {
        'threshold': float(level_data['threshold']),
        'num_events': int(level_data['num_events']),
        'cluster_labels': level_data['labels'].tolist()
    }

hierarchy_json_path = output_dir / f"{video_name}_hierarchy.json"
with open(hierarchy_json_path, 'w', encoding='utf-8') as f:
    json.dump(hierarchy_json, f, ensure_ascii=False, indent=2)

print(f"✅ 层次结构JSON: {hierarchy_json_path}")

# %% [markdown]
# ## 📊 最终摘要

# %%
print("\n" + "="*70)
print(f"📊 玩具车拆装分割结果 - {video_name}")
print("="*70)
print(f"视频时长: {T/FPS:.2f}秒 ({T}帧), FPS: {FPS}")
print()

print("【原子级分割】")
print(f"  方法: 非对称滑动窗口")
print(f"  边界数: {len(boundaries)}")
print(f"  片段数: {len(segs)}")
print(f"  平均片段长度: {params['avg_segment_sec']:.1f}秒")
print(f"  质量分数: {params['final_score']:.3f}")
print()

print("【层次结构】")
for level_name in level_names:
    level_data = hierarchical_segmentations[level_name]
    print(f"  {level_name:12s}: {level_data['num_events']:3d} 个事件")

print()
print("💡 解释:")
print("  - ATOMIC    : 拆/装单个螺丝、取下/安装单个小部件")
print("  - COMPONENT : 拆/装单个轮子、车身等完整部件")
print("  - SIDE      : 拆/装左侧、右侧（如左右轮子组合）")
print("  - ACTIVITY  : 最顶层：拆解玩具车 vs 组装玩具车")
print()
print(f"📁 所有输出已保存至: {output_dir}")
print("   - *_atomic_segmentation.json (原子分割)")
print("   - *_atomic_segments.png (原子分割可视化)")
print("   - *_hierarchy.json (层次结构)")
print("   - *_SCM.png (语义一致性图谱)")
print("   - *_dendrogram.png (树状图)")
print("   - *_hierarchy.png (多粒度分割)")
print()
print("🎉 Pipeline执行完成!")
print("="*70)

✅ PyTorch: 2.10.0+cu128, CUDA: True
GPU: NVIDIA GeForce RTX 5090

🎬 视频: C10390_rgb
📁 输出: pipeline_output/C10390_rgb
🔧 设备: cuda, FPS: 10
🎯 策略: 完整增强版（不对称窗口 + 差分峰值 + 融合 + 精修）

[Step 1/3] 帧提取
使用FFmpeg提取帧...


/media/wang/New Volume1/josie/gebd/C10390_rgb.mp4: No such file or directory


CalledProcessError: Command '['ffmpeg', '-y', '-v', 'error', '-i', '/media/wang/New Volume1/josie/gebd/C10390_rgb.mp4', '-vf', 'fps=10', '-q:v', '2', 'pipeline_output/C10390_rgb/frames/%06d.jpg']' returned non-zero exit status 1.

In [ ]:
# ======================================================================
# 参数倒推测试：根据目标分割时间点寻找最佳参数
# ======================================================================

import itertools
from scipy.cluster.hierarchy import linkage, fcluster

print("="*70)
print("🔍 参数倒推测试")
print("="*70)
print()

# 目标分割时间点（秒）
target_splits_top = [148]  # 2:28 - 顶层分割点（2个事件）
target_splits_lower = [8, 31, 52, 64, 148, 160, 180, 208, 225, 251, 278]  # 下层分割点

print("🎯 目标分割时间点:")
print(f"  顶层 (2个事件): {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
print(f"  下层 ({len(target_splits_lower)+1}个事件): ", end="")
for t in target_splits_lower:
    print(f"{t//60:02d}:{t%60:02d}", end=" ")
print("\n")

# 参数搜索范围（大幅扩大）
ws_range = range(5, 35, 3)  # 滑动窗口大小: 5, 8, 11, ..., 32
thr_range = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # 阈值
sigma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120]  # SCM时间核带宽
tolerance = 20  # 允许的时间误差（秒）- 放宽到20秒

print("🔧 参数搜索范围:")
print(f"  滑动窗口大小 (ws): {list(ws_range)}")
print(f"  阈值 (thr): {thr_range}")
print(f"  SCM时间核带宽 (sigma): {sigma_range}")
print(f"  容差: ±{tolerance}秒")
print()

# 先做一个快速测试，看看当前参数能产生什么结果
print("🧪 快速测试当前参数...")
try:
    # 使用notebook中已有的变量
    print(f"  当前原子分割产生 {len(segs)} 个片段")
    print(f"  原子单元数: {K}")
    
    # 重新计算层次聚类（因为这是新cell）
    test_cos = atomic_features @ atomic_features.T
    test_time_diff = np.abs(atomic_centers[:, None] - atomic_centers[None, :])
    test_temporal = np.exp(-test_time_diff**2 / (2 * 50**2))
    test_SCM = test_cos * test_temporal
    test_dist = 1 - test_SCM
    np.fill_diagonal(test_dist, 0)
    test_condensed = test_dist[np.triu_indices(K, k=1)]
    test_Z = linkage(test_condensed, method='ward')
    
    # 测试2个事件
    test_labels_2 = fcluster(test_Z, t=2, criterion='maxclust')
    test_splits_2 = [int(atomic_centers[i]) for i in range(1, K) if test_labels_2[i] != test_labels_2[i-1]]
    print(f"  2个事件的分割点: ", end="")
    for t in test_splits_2:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    
    # 测试12个事件
    test_labels_12 = fcluster(test_Z, t=12, criterion='maxclust')
    test_splits_12 = [int(atomic_centers[i]) for i in range(1, K) if test_labels_12[i] != test_labels_12[i-1]]
    print(f"  12个事件的分割点: ", end="")
    for t in test_splits_12:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print()
except Exception as e:
    print(f"  测试失败: {e}")
    import traceback
    traceback.print_exc()
    print()

def time_to_frame(time_sec, fps=FPS):
    """将时间（秒）转换为帧索引"""
    return int(time_sec * fps)

def calculate_split_error(actual_splits, target_splits, tolerance=15):
    """计算实际分割点与目标分割点的误差（允许数量不完全匹配）"""
    if len(actual_splits) == 0 or len(target_splits) == 0:
        return float('inf')
    
    # 允许数量有较大差异（放宽到5）
    if abs(len(actual_splits) - len(target_splits)) > 5:
        return float('inf')
    
    # 使用最近邻匹配
    total_error = 0
    matched_count = 0
    
    for target in target_splits:
        # 找到最接近的实际分割点
        min_dist = float('inf')
        for actual in actual_splits:
            dist = abs(actual - target)
            if dist < min_dist:
                min_dist = dist
        
        if min_dist <= tolerance:  # 在容差范围内
            total_error += min_dist
            matched_count += 1
    
    if matched_count == 0:
        return float('inf')
    
    # 惩罚未匹配的分割点
    penalty = abs(len(actual_splits) - len(target_splits)) * 10
    return (total_error / matched_count) + penalty

def get_split_times_from_segments(segments, fps=FPS):
    """从分割结果中提取分割时间点（秒）"""
    split_times = []
    for i in range(len(segments) - 1):
        split_frame = segments[i][1]  # 当前片段的结束帧
        split_time = split_frame / fps
        split_times.append(int(split_time))
    return split_times

# 准备归一化特征（与notebook中一致）
feats_unit = features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-8)
print(f"✅ 特征归一化完成: {feats_unit.shape}")
print()

# 先测试几个参数组合，看看能产生多少片段
print("🧪 测试几个参数组合...")
scaler_test = StandardScaler()
feats_scaled_test = scaler_test.fit_transform(features)
test_params = [(10, 0.5), (15, 0.6), (20, 0.7), (25, 0.8)]
for ws, thr in test_params:
    try:
        bounds, _, _ = sw_asymmetric(feats_scaled_test, window_size=ws, threshold=thr, 
                                      similarity_method='cosine', min_segment_frames=max(FPS, ws))
        print(f"  ws={ws}, thr={thr:.1f}: {len(bounds)} 个边界 -> {len(bounds)+1} 个片段")
    except Exception as e:
        print(f"  ws={ws}, thr={thr:.1f}: 错误 - {e}")
print()

print("🔄 开始参数搜索...")
print()

best_params = None
best_error = float('inf')
best_results = None
all_results = []  # 保存所有有效结果
error_count = 0  # 统计异常次数
debug_info = []  # 调试信息

total_combinations = len(ws_range) * len(thr_range) * len(sigma_range)
tested = 0

for ws, thr, sigma in itertools.product(ws_range, thr_range, sigma_range):
    tested += 1
    if tested % 50 == 0:
        print(f"  已测试 {tested}/{total_combinations} 组参数... (找到 {len(all_results)} 个有效组合)")
    
    try:
        # 1. 原子分割 - 使用与实际运行相同的完整方法
        # 标准化特征
        scaler_loop = StandardScaler()
        feats_scaled_loop = scaler_loop.fit_transform(features)
        
        # 步骤1: 不对称滑动窗口
        min_seg = max(FPS, ws)
        sw_boundaries, _, _ = sw_asymmetric(
            feats_scaled_loop, window_size=ws, step_reset=1,
            threshold=thr, similarity_method='cosine', min_segment_frames=min_seg
        )
        
        # 步骤2: 差分峰值检测
        div_boundaries = traditional_diff_peaks(feats_scaled_loop, FPS)
        
        # 步骤3: 融合与精修（使用与enhanced_runner相同的逻辑）
        boundaries_test = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled_loop, FPS)
        
        # 转换为片段格式
        segs_test = []
        prev = 0
        for b in boundaries_test:
            segs_test.append((prev, b))
            prev = b + 1
        segs_test.append((prev, features.shape[0] - 1))
        
        if len(segs_test) < 2:
            if tested <= 10:  # 记录更多
                debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 分割数太少 ({len(segs_test)})")
            continue
        
        # 记录一些成功的案例
        if tested <= 10 and len(segs_test) >= 10:
            debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 成功产生 {len(segs_test)} 个片段")
        
        # 2. 构建原子特征和中心点
        atomic_features_test = []
        atomic_centers_test = []
        for start, end in segs_test:
            seg_feat = features[start:end+1].mean(axis=0)
            atomic_features_test.append(seg_feat)
            atomic_centers_test.append((start + end) / 2 / FPS)
        
        atomic_features_test = np.array(atomic_features_test)
        atomic_centers_test = np.array(atomic_centers_test)
        K_test = len(segs_test)
        
        # 3. 构建SCM
        cos_sim_test = atomic_features_test @ atomic_features_test.T
        time_diff_test = np.abs(atomic_centers_test[:, None] - atomic_centers_test[None, :])
        temporal_kernel_test = np.exp(-time_diff_test**2 / (2 * sigma**2))
        SCM_test = cos_sim_test * temporal_kernel_test
        
        # 4. 层次聚类
        dist_matrix_test = 1 - SCM_test
        np.fill_diagonal(dist_matrix_test, 0)
        condensed_dist_test = dist_matrix_test[np.triu_indices(K_test, k=1)]
        Z_test = linkage(condensed_dist_test, method='ward')
        
        # 测试顶层（2个事件）
        labels_top = fcluster(Z_test, t=2, criterion='maxclust')
        
        # 找到顶层分割点
        top_splits = []
        for i in range(1, K_test):
            if labels_top[i] != labels_top[i-1]:
                split_time = int(atomic_centers_test[i])
                top_splits.append(split_time)
        
        # 计算顶层误差
        top_error = calculate_split_error(top_splits, target_splits_top, tolerance)
        
        # 测试多个下层事件数（允许一定范围）
        best_lower_error = float('inf')
        best_lower_splits = []
        best_num_events = 0
        
        for num_events_lower in range(8, 16):  # 尝试8-15个事件
            try:
                labels_lower = fcluster(Z_test, t=num_events_lower, criterion='maxclust')
                
                # 找到下层分割点
                lower_splits = []
                for i in range(1, K_test):
                    if labels_lower[i] != labels_lower[i-1]:
                        split_time = int(atomic_centers_test[i])
                        lower_splits.append(split_time)
                
                # 计算下层误差
                lower_error = calculate_split_error(lower_splits, target_splits_lower, tolerance)
                
                if lower_error < best_lower_error:
                    best_lower_error = lower_error
                    best_lower_splits = lower_splits
                    best_num_events = num_events_lower
            except:
                continue
        
        if best_lower_error == float('inf'):
            if len(all_results) < 5:  # 记录为什么被过滤
                debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 下层误差过大 (top_error={top_error:.1f})")
            continue
        
        lower_error = best_lower_error
        lower_splits = best_lower_splits
        
        # 记录找到的有效组合
        if len(all_results) < 5:
            debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 找到有效组合! top_err={top_error:.1f}, lower_err={lower_error:.1f}")
        
        # 总误差（加权）
        total_error = top_error * 3 + lower_error  # 顶层误差权重更高
        
        # 保存所有有效结果
        all_results.append({
            'params': {'ws': ws, 'thr': thr, 'sigma': sigma},
            'top_splits': top_splits,
            'lower_splits': lower_splits,
            'top_error': top_error,
            'lower_error': lower_error,
            'total_error': total_error,
            'num_atomic': K_test,
            'num_events_lower': best_num_events
        })
        
        if total_error < best_error:
            best_error = total_error
            best_params = {'ws': ws, 'thr': thr, 'sigma': sigma}
            best_results = {
                'top_splits': top_splits,
                'lower_splits': lower_splits,
                'top_error': top_error,
                'lower_error': lower_error,
                'num_atomic': K_test
            }
    
    except Exception as e:
        error_count += 1
        if error_count <= 5:  # 只记录前5个错误
            debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 异常 - {str(e)[:50]}")
        continue

print()
print("="*70)
print("✅ 参数搜索完成！")
print("="*70)
print()

print(f"📊 统计信息:")
print(f"  测试组合数: {tested}")
print(f"  异常次数: {error_count}")
print(f"  有效组合数: {len(all_results)}")
print()

if debug_info:
    print("🔍 调试信息（前几次）:")
    for info in debug_info[:10]:
        print(f"  {info}")
    print()

if best_params is not None:
    print("🎯 最佳参数:")
    print(f"  滑动窗口大小 (ws): {best_params['ws']}")
    print(f"  阈值 (thr): {best_params['thr']:.2f}")
    print(f"  SCM时间核带宽 (sigma): {best_params['sigma']}秒")
    print(f"  原子单元数量: {best_results['num_atomic']}")
    print()
    
    print("📊 分割结果对比:")
    print()
    print("  顶层 (2个事件):")
    print(f"    目标分割点: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
    if best_results['top_splits']:
        for t in best_results['top_splits']:
            print(f"    实际分割点: {t//60:02d}:{t%60:02d}")
        print(f"    误差: {best_results['top_error']:.1f}秒")
    print()
    
    print(f"  下层 ({len(best_results['lower_splits'])+1}个事件):")
    print("    目标分割点: ", end="")
    for t in target_splits_lower:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print("    实际分割点: ", end="")
    for t in best_results['lower_splits']:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print(f"    平均误差: {best_results['lower_error']:.1f}秒")
    print()
    
    print("💡 建议:")
    print(f"  将notebook中的参数修改为:")
    print(f"    - 滑动窗口: ws={best_params['ws']}, thr={best_params['thr']:.2f}")
    print(f"    - SCM: sigma={best_params['sigma']}")
    print(f"    - 层次聚类: 顶层2个事件，下层约{len(best_results['lower_splits'])+1}个事件")
else:
    print("❌ 未找到完全匹配的参数组合")
    print()
    
    # 显示前10个最接近的结果
    if all_results:
        print("📋 最接近的10组参数（按误差排序）:")
        print()
        all_results_sorted = sorted(all_results, key=lambda x: x['total_error'])
        
        for i, result in enumerate(all_results_sorted[:10], 1):
            params = result['params']
            print(f"  {i}. ws={params['ws']}, thr={params['thr']:.2f}, sigma={params['sigma']}")
            print(f"     原子单元: {result['num_atomic']}个, 下层事件: {result['num_events_lower']}个")
            print(f"     顶层误差: {result['top_error']:.1f}秒, 下层误差: {result['lower_error']:.1f}秒")
            print(f"     顶层分割: ", end="")
            for t in result['top_splits']:
                print(f"{t//60:02d}:{t%60:02d}", end=" ")
            print(f" (目标: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d})")
            print(f"     下层分割: ", end="")
            for t in result['lower_splits']:
                print(f"{t//60:02d}:{t%60:02d}", end=" ")
            print()
            print()
    else:
        print("   没有找到任何有效的参数组合")
        print("   建议:")
        print("   1. 进一步扩大搜索范围")
        print("   2. 检查目标分割点是否与视频内容匹配")
        print(\"   3. 调整目标分割点的容差\")\n",
    "\n",
    "print()\n",
    "print(\"=\"*70)"

In [11]:
# ======================================================================
# 保存最优参数的分割结果（固定使用 ws=14, thr=0.50, sigma=80）
# ======================================================================

print("="*70)
print("💾 保存最优参数的分割结果")
print("="*70)
print()

# 使用改进版参数倒推测试找到的最优参数（实际边界在02:29）
ws = 23
thr = 0.80
sigma = 100
print(f"📋 最优参数（匹配实际边界）: ws={ws}, thr={thr:.2f}, sigma={sigma}")
print("📋 这组参数产生的实际边界在02:29（非常接近目标02:28！）")
print()

# 重新运行以获取完整数据（与参数倒推测试完全一致的方法）
scaler_save = StandardScaler()
feats_scaled_save = scaler_save.fit_transform(features)

min_seg = max(FPS, ws)
sw_boundaries_save, _, _ = sw_asymmetric(
    feats_scaled_save, window_size=ws, step_reset=1,
    threshold=thr, similarity_method='cosine', min_segment_frames=min_seg
)

div_boundaries_save = traditional_diff_peaks(feats_scaled_save, FPS)
boundaries_save = post_fuse_and_refine(sw_boundaries_save, div_boundaries_save, feats_scaled_save, FPS)

# 转换为片段格式（与参数倒推测试一致）
segs_save = []
prev = 0
for b in boundaries_save:
    segs_save.append((prev, b))
    prev = b + 1
segs_save.append((prev, features.shape[0] - 1))

print(f"✓ 原子片段: {len(segs_save)}个")

# 构建原子特征和中心点（与参数倒推测试一致）
atomic_features_save = []
atomic_centers_save = []
for start, end in segs_save:
    seg_feat = features[start:end+1].mean(axis=0)  # 闭区间
    atomic_features_save.append(seg_feat)  # 不归一化
    atomic_centers_save.append((start + end) / 2 / FPS)

atomic_features_save = np.array(atomic_features_save)
atomic_centers_save = np.array(atomic_centers_save)
K_save = len(segs_save)

print(f"✓ 原子特征: {atomic_features_save.shape}")

# 构建SCM（与参数倒推测试一致）
cos_sim_save = atomic_features_save @ atomic_features_save.T
time_diff_save = np.abs(atomic_centers_save[:, None] - atomic_centers_save[None, :])
temporal_kernel_save = np.exp(-time_diff_save**2 / (2 * sigma**2))
SCM_save = cos_sim_save * temporal_kernel_save

print(f"✓ SCM: {SCM_save.shape}")

# 层次聚类（与参数倒推测试一致）
dist_matrix_save = 1 - SCM_save
np.fill_diagonal(dist_matrix_save, 0)
condensed_dist_save = dist_matrix_save[np.triu_indices(K_save, k=1)]
Z_save = linkage(condensed_dist_save, method='ward')

print(f"✓ 聚类矩阵: {Z_save.shape}")
print()

# 验证分割点
labels_verify = fcluster(Z_save, t=2, criterion='maxclust')
verify_splits = []
for i in range(1, K_save):
    if labels_verify[i] != labels_verify[i-1]:
        split_time = int(atomic_centers_save[i])
        verify_splits.append(split_time)

print("🔍 验证分割点（原子中心点）:")
for t in verify_splits:
    print(f"  {t//60:02d}:{t%60:02d} ({t}秒)")
print()

# 保存所有结果
np.save('optimal_segs.npy', np.array(segs_save, dtype=object))
np.save('optimal_atomic_features.npy', atomic_features_save)
np.save('optimal_atomic_centers.npy', atomic_centers_save)
np.save('optimal_SCM.npy', SCM_save)
np.save('optimal_linkage.npy', Z_save)
np.save('optimal_params.npy', np.array([ws, thr, sigma]))

print("="*70)
print("✅ 保存完成！")
print("="*70)
print()
print("📁 保存的文件:")
print("  • optimal_segs.npy - 原子片段")
print("  • optimal_atomic_features.npy - 原子特征")
print("  • optimal_atomic_centers.npy - 原子中心点")
print("  • optimal_SCM.npy - 语义一致性图谱")
print("  • optimal_linkage.npy - 聚类矩阵")
print("  • optimal_params.npy - 参数")
print()

# 输出详细的分割时间点
print("="*70)
print("📋 详细分割时间点")
print("="*70)
print()
    
def format_time(seconds):
    return f"{int(seconds//60):02d}:{int(seconds%60):02d}"

def labels_to_segments(labels, atomic_segs):
    segments = []
    K = len(labels)
    event_boundaries = [0]
    for i in range(1, K):
        if labels[i] != labels[i-1]:
            event_boundaries.append(i)
    event_boundaries.append(K)
    for i in range(len(event_boundaries) - 1):
        start_idx = event_boundaries[i]
        end_idx = event_boundaries[i+1] - 1
        start_frame = atomic_segs[start_idx][0]
        end_frame = atomic_segs[end_idx][1]
        segments.append((start_frame, end_frame))
    return segments

# 生成多粒度分割
hierarchical_segs = {}
hierarchical_segs['activity'] = labels_to_segments(fcluster(Z_save, t=2, criterion='maxclust'), segs_save)
hierarchical_segs['side'] = labels_to_segments(fcluster(Z_save, t=4, criterion='maxclust'), segs_save)
hierarchical_segs['component'] = labels_to_segments(fcluster(Z_save, t=8, criterion='maxclust'), segs_save)

for level_name, icon, title in zip(
    ['activity', 'side', 'component'],
    ['🔴', '🟠', '🟢'],
    ['ACTIVITY', 'SIDE', 'COMPONENT']
):
    segs_level = hierarchical_segs[level_name]
    num_events = len(segs_level)
    print(f"{icon} {title}层 - {num_events} 个事件")
    print("="*70)
    for i, (start_frame, end_frame) in enumerate(segs_level):
        start_time = start_frame / FPS
        end_time = end_frame / FPS
        duration_event = end_time - start_time
        print(f"  Event {i+1}: {format_time(start_time)} - {format_time(end_time)}  "
                f"(时长: {duration_event:6.1f}秒, {int(duration_event/60):2d}分{int(duration_event%60):02d}秒)")
    if num_events > 1:
        boundary_times = [seg[1] / FPS for seg in segs_level[:-1]]
        print(f"\n  🔹 分割点 ({len(boundary_times)}个): {', '.join([format_time(t) for t in boundary_times])}")
    print("\n")

    print("="*70)
    print("💡 现在可以在可视化notebook中加载这些结果")
    print()

💾 保存最优参数的分割结果

📋 最优参数（匹配实际边界）: ws=23, thr=0.80, sigma=100
📋 这组参数产生的实际边界在02:29（非常接近目标02:28！）

✓ 原子片段: 142个
✓ 原子特征: (142, 512)
✓ SCM: (142, 142)
✓ 聚类矩阵: (141, 4)

🔍 验证分割点（原子中心点）:
  02:31 (151秒)

✅ 保存完成！

📁 保存的文件:
  • optimal_segs.npy - 原子片段
  • optimal_atomic_features.npy - 原子特征
  • optimal_atomic_centers.npy - 原子中心点
  • optimal_SCM.npy - 语义一致性图谱
  • optimal_linkage.npy - 聚类矩阵
  • optimal_params.npy - 参数

📋 详细分割时间点

🔴 ACTIVITY层 - 2 个事件
  Event 1: 00:00 - 02:29  (时长:  149.5秒,  2分29秒)
  Event 2: 02:29 - 04:38  (时长:  128.8秒,  2分08秒)

  🔹 分割点 (1个): 02:29


💡 现在可以在可视化notebook中加载这些结果

🟠 SIDE层 - 4 个事件
  Event 1: 00:00 - 01:20  (时长:   80.2秒,  1分20秒)
  Event 2: 01:20 - 02:29  (时长:   69.2秒,  1分09秒)
  Event 3: 02:29 - 03:35  (时长:   65.4秒,  1分05秒)
  Event 4: 03:35 - 04:38  (时长:   63.3秒,  1分03秒)

  🔹 分割点 (3个): 01:20, 02:29, 03:35


💡 现在可以在可视化notebook中加载这些结果

🟢 COMPONENT层 - 8 个事件
  Event 1: 00:00 - 00:31  (时长:   31.5秒,  0分31秒)
  Event 2: 00:31 - 00:52  (时长:   20.5秒,  0分20秒)
  Event 3: 00:52 - 01:20  (

In [ ]:
# ======================================================================
# 参数倒推测试（改进版）：匹配实际边界帧而不是原子中心点
# ======================================================================

import itertools
from scipy.cluster.hierarchy import linkage, fcluster

print("="*70)
print("🔍 参数倒推测试（改进版 - 匹配实际边界帧）")
print("="*70)
print()

# 目标分割时间点（秒）- 这次我们要匹配实际边界
target_splits_top = [148]  # 2:28 - 顶层分割点（2个事件）
target_splits_lower = [8, 31, 52, 64, 148, 160, 180, 208, 225, 251, 278]  # 下层分割点

print("🎯 目标分割时间点（实际边界）:")
print(f"  顶层 (2个事件): {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
print(f"  下层 ({len(target_splits_lower)+1}个事件): ", end="")
for t in target_splits_lower:
    print(f"{t//60:02d}:{t%60:02d}", end=" ")
print("\n")

# 参数搜索范围
ws_range = range(5, 35, 3)
thr_range = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sigma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120]
tolerance = 20  # 允许的时间误差（秒）

print("🔧 参数搜索范围:")
print(f"  滑动窗口大小 (ws): {list(ws_range)}")
print(f"  阈值 (thr): {thr_range}")
print(f"  SCM时间核带宽 (sigma): {sigma_range}")
print(f"  容差: ±{tolerance}秒")
print()

def calculate_split_error(actual_splits, target_splits, tolerance=15):
    """计算实际分割点与目标分割点的误差"""
    if len(actual_splits) == 0 or len(target_splits) == 0:
        return float('inf')
    
    if abs(len(actual_splits) - len(target_splits)) > 5:
        return float('inf')
    
    total_error = 0
    matched_count = 0
    
    for target in target_splits:
        min_dist = float('inf')
        for actual in actual_splits:
            dist = abs(actual - target)
            if dist < min_dist:
                min_dist = dist
        
        if min_dist <= tolerance:
            total_error += min_dist
            matched_count += 1
    
    if matched_count == 0:
        return float('inf')
    
    penalty = abs(len(actual_splits) - len(target_splits)) * 10
    return (total_error / matched_count) + penalty

print("🔄 开始参数搜索（匹配实际边界帧）...")
print()

best_params_boundary = None
best_error_boundary = float('inf')
best_results_boundary = None
all_results_boundary = []
error_count = 0

total_combinations = len(ws_range) * len(thr_range) * len(sigma_range)
tested = 0

for ws, thr, sigma in itertools.product(ws_range, thr_range, sigma_range):
    tested += 1
    if tested % 50 == 0:
        print(f"  已测试 {tested}/{total_combinations} 组参数... (找到 {len(all_results_boundary)} 个有效组合)")
    
    try:
        # 1. 原子分割
        scaler_loop = StandardScaler()
        feats_scaled_loop = scaler_loop.fit_transform(features)
        
        min_seg = max(FPS, ws)
        sw_boundaries, _, _ = sw_asymmetric(
            feats_scaled_loop, window_size=ws, step_reset=1,
            threshold=thr, similarity_method='cosine', min_segment_frames=min_seg
        )
        
        div_boundaries = traditional_diff_peaks(feats_scaled_loop, FPS)
        boundaries_test = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled_loop, FPS)
        
        # 转换为片段格式
        segs_test = []
        prev = 0
        for b in boundaries_test:
            segs_test.append((prev, b))
            prev = b + 1
        segs_test.append((prev, features.shape[0] - 1))
        
        if len(segs_test) < 2:
            continue
        
        # 2. 构建原子特征和中心点
        atomic_features_test = []
        atomic_centers_test = []
        for start, end in segs_test:
            seg_feat = features[start:end+1].mean(axis=0)
            atomic_features_test.append(seg_feat)
            atomic_centers_test.append((start + end) / 2 / FPS)
        
        atomic_features_test = np.array(atomic_features_test)
        atomic_centers_test = np.array(atomic_centers_test)
        K_test = len(segs_test)
        
        # 3. 构建SCM
        cos_sim_test = atomic_features_test @ atomic_features_test.T
        time_diff_test = np.abs(atomic_centers_test[:, None] - atomic_centers_test[None, :])
        temporal_kernel_test = np.exp(-time_diff_test**2 / (2 * sigma**2))
        SCM_test = cos_sim_test * temporal_kernel_test
        
        # 4. 层次聚类
        dist_matrix_test = 1 - SCM_test
        np.fill_diagonal(dist_matrix_test, 0)
        condensed_dist_test = dist_matrix_test[np.triu_indices(K_test, k=1)]
        Z_test = linkage(condensed_dist_test, method='ward')
        
        # 测试顶层（2个事件）- 使用实际边界帧
        labels_top = fcluster(Z_test, t=2, criterion='maxclust')
        
        # 找到顶层分割点（使用实际边界帧而不是中心点）
        top_splits = []
        for i in range(1, K_test):
            if labels_top[i] != labels_top[i-1]:
                boundary_frame = segs_test[i][0]  # 使用起始帧
                boundary_time = int(boundary_frame / FPS)
                top_splits.append(boundary_time)
        
        # 计算顶层误差
        top_error = calculate_split_error(top_splits, target_splits_top, tolerance)
        
        # 测试多个下层事件数
        best_lower_error = float('inf')
        best_lower_splits = []
        best_num_events = 0
        
        for num_events_lower in range(8, 16):
            try:
                labels_lower = fcluster(Z_test, t=num_events_lower, criterion='maxclust')
                
                # 找到下层分割点（使用实际边界帧）
                lower_splits = []
                for i in range(1, K_test):
                    if labels_lower[i] != labels_lower[i-1]:
                        boundary_frame = segs_test[i][0]  # 使用起始帧
                        boundary_time = int(boundary_frame / FPS)
                        lower_splits.append(boundary_time)
                
                # 计算下层误差
                lower_error = calculate_split_error(lower_splits, target_splits_lower, tolerance)
                
                if lower_error < best_lower_error:
                    best_lower_error = lower_error
                    best_lower_splits = lower_splits
                    best_num_events = num_events_lower
            except:
                continue
        
        if best_lower_error == float('inf'):
            continue
        
        lower_error = best_lower_error
        lower_splits = best_lower_splits
        
        # 总误差
        total_error = top_error * 3 + lower_error
        
        # 保存所有有效结果
        all_results_boundary.append({
            'params': {'ws': ws, 'thr': thr, 'sigma': sigma},
            'top_splits': top_splits,
            'lower_splits': lower_splits,
            'top_error': top_error,
            'lower_error': lower_error,
            'total_error': total_error,
            'num_atomic': K_test,
            'num_events_lower': best_num_events
        })
        
        if total_error < best_error_boundary:
            best_error_boundary = total_error
            best_params_boundary = {'ws': ws, 'thr': thr, 'sigma': sigma}
            best_results_boundary = {
                'top_splits': top_splits,
                'lower_splits': lower_splits,
                'top_error': top_error,
                'lower_error': lower_error,
                'num_atomic': K_test
            }
    
    except Exception as e:
        error_count += 1
        continue

print()
print("="*70)
print("✅ 参数搜索完成（匹配实际边界帧）！")
print("="*70)
print()

print(f"📊 统计信息:")
print(f"  测试组合数: {tested}")
print(f"  异常次数: {error_count}")
print(f"  有效组合数: {len(all_results_boundary)}")
print()

if best_params_boundary is not None:
    print("🎯 最佳参数（匹配实际边界帧）:")
    print(f"  滑动窗口大小 (ws): {best_params_boundary['ws']}")
    print(f"  阈值 (thr): {best_params_boundary['thr']:.2f}")
    print(f"  SCM时间核带宽 (sigma): {best_params_boundary['sigma']}秒")
    print(f"  原子单元数量: {best_results_boundary['num_atomic']}")
    print()
    
    print("📊 分割结果对比（实际边界帧）:")
    print()
    print("  顶层 (2个事件):")
    print(f"    目标分割点: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
    if best_results_boundary['top_splits']:
        for t in best_results_boundary['top_splits']:
            print(f"    实际分割点: {t//60:02d}:{t%60:02d}")
        print(f"    误差: {best_results_boundary['top_error']:.1f}秒")
    print()
    
    print(f"  下层 ({len(best_results_boundary['lower_splits'])+1}个事件):")
    print("    目标分割点: ", end="")
    for t in target_splits_lower:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print("    实际分割点: ", end="")
    for t in best_results_boundary['lower_splits']:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print(f"    平均误差: {best_results_boundary['lower_error']:.1f}秒")
    print()
    
    print("💡 对比:")
    print("  原版（匹配原子中心点）: ws=14, thr=0.50, sigma=80 -> 中心点02:27, 边界02:11")
    print(f"  改进版（匹配实际边界）: ws={best_params_boundary['ws']}, thr={best_params_boundary['thr']:.2f}, sigma={best_params_boundary['sigma']} -> 边界{best_results_boundary['top_splits'][0]//60:02d}:{best_results_boundary['top_splits'][0]%60:02d}")
else:
    print("❌ 未找到完全匹配的参数组合")
    
    if all_results_boundary:
        print()
        print("📋 最接近的10组参数（按误差排序）:")
        print()
        all_results_sorted = sorted(all_results_boundary, key=lambda x: x['total_error'])
        
        for i, result in enumerate(all_results_sorted[:10], 1):
            params = result['params']
            print(f"  {i}. ws={params['ws']}, thr={params['thr']:.2f}, sigma={params['sigma']}")
            print(f"     原子单元: {result['num_atomic']}个")
            print(f"     顶层误差: {result['top_error']:.1f}秒, 下层误差: {result['lower_error']:.1f}秒")
            print(f"     顶层分割: ", end="")
            for t in result['top_splits']:
                print(f"{t//60:02d}:{t%60:02d}", end=" ")
            print(f" (目标: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d})")
            print()

print()